# TID — Veri Cikarim Notebook'u
**Hazırlayan:** Eren Reyhanlioglu  
**Tarih:** Mayis 2026

## Ne Yapiyor?
AUTSL veri setindeki ham color (.mp4) videolarından MediaPipe Holistic ile
iskelet koordinatları cikarip `.npy` formatinda Drive'a kaydeder.

## Veri Seti
- AUTSL (Ankara University Turkish Sign Language Dataset)
- 226 isaretci, 43 farklı katılımcı — biz 30 kelime kullandik (label 0-29)
- Orjinal indirme: https://data.chalearnlap.cvc.uab.cat/AuTSL/data/

## Sifreler
| Dosya | Sifre |
|-------|-------|
| Train video zip | MdG3z6Eh1t |
| Val video zip | bhRY5B9zS2 |
| Val labels zip | zYX5W7fZ |
| Test video zip | ds6Kvdus3o |
| Test labels zip | ds6Kvdus3o |

Not: Test zip'i bozuk indirilmisti. Silip yeniden indirmek gerekti.
Drive'a direkt yazılamiyor — once /content/ a indir, sonra shutil.copy ile Drive'a kopyala.

## Cikarilan Ozellikler (170 toplam)
Her video frame'i (170,) boyutlu vektor:
- Pose: 11 nokta x 4 = 44 ozellik  
  Indeksler: 0 (burun), 7-8 (kulaklar), 11-16 (omuz/dirsek/bilek), 23-24 (kalca)
- Sol el: 21 nokta x 3 = 63 ozellik
- Sag el: 21 nokta x 3 = 63 ozellik

## Cikti
ErenDataset/
    train/  — 3696 .npy dosyasi (30 sinif, ~123/sinif)
    val/    — 578  .npy dosyasi (30 sinif, ~19/sinif)
    test/   — 494  .npy dosyasi (30 sinif, ~16/sinif)

Her .npy dosyasi: (frame_sayisi, 170) shape, float32

## MediaPipe Kurulum Notu
TensorFlow ile protobuf cakismasi var. Su sirayla yukle:
1. pip install mediapipe==0.10.14
2. Kernel restart et
3. import sys; sys.modules['tensorflow'] = None — sonra mediapipe'i import et

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile

zip_path = '/content/drive/MyDrive/Nöron-Ağları-Proje/MVP_Dataset.zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    names = z.namelist()
    print(f"Toplam dosya sayısı: {len(names)}")
    print("\nİlk 20 dosya:")
    for n in names[:20]:
        print(n)

Mounted at /content/drive
Toplam dosya sayısı: 4596

İlk 20 dosya:
content/MVP_Dataset/
content/MVP_Dataset/test/
content/MVP_Dataset/test/6_hayir/
content/MVP_Dataset/test/6_hayir/signer27_sample25_color.mp4
content/MVP_Dataset/test/6_hayir/signer27_sample391_depth.mp4
content/MVP_Dataset/test/6_hayir/signer30_sample208_color.mp4
content/MVP_Dataset/test/6_hayir/signer14_sample229_depth.mp4
content/MVP_Dataset/test/6_hayir/signer14_sample363_color.mp4
content/MVP_Dataset/test/6_hayir/signer27_sample189_color.mp4
content/MVP_Dataset/test/6_hayir/signer30_sample431_depth.mp4
content/MVP_Dataset/test/20_hastane/
content/MVP_Dataset/test/20_hastane/signer30_sample437_color.mp4
content/MVP_Dataset/test/20_hastane/signer27_sample374_depth.mp4
content/MVP_Dataset/test/20_hastane/signer30_sample490_depth.mp4
content/MVP_Dataset/test/20_hastane/signer14_sample118_color.mp4
content/MVP_Dataset/test/20_hastane/signer27_sample262_color.mp4
content/MVP_Dataset/test/20_hastane/signer14_sample257_co

## **Veri Seti İncelemesi**

In [ ]:
from collections import defaultdict

color_files = [n for n in names if 'color.mp4' in n]
print(f"Toplam color video: {len(color_files)}")

counts = defaultdict(lambda: defaultdict(int))
for f in color_files:
    parts = f.split('/')
    if len(parts) >= 4:
        split = parts[2]
        cls   = parts[3]
        counts[split][cls] += 1

for split in ['train', 'val', 'test']:
    total = sum(counts[split].values())
    print(f"\n{split.upper()} — {total} video, {len(counts[split])} sınıf")
    for cls, cnt in sorted(counts[split].items()):
        print(f"  {cls:25s} {cnt}")

Toplam color video: 2282

TRAIN — 1899 video, 30 sınıf
  0_ben                     66
  10_var                    59
  11_yok                    66
  12_iyi                    62
  13_kotu                   58
  14_nasil                  61
  15_neden                  62
  16_nerede                 64
  17_yardim                 68
  18_doktor                 59
  19_hasta                  64
  1_sen                     69
  20_hastane                65
  21_ilac                   65
  22_gecmis_olsun           66
  23_ataturk                55
  24_ev                     61
  25_zaman                  64
  26_icmek                  55
  27_yemek                  66
  28_yapmak                 63
  29_bakmak                 70
  2_selam                   70
  3_hoscakal                62
  4_tamam                   48
  5_evet                    69
  6_hayir                   64
  7_tesekkur                63
  8_rica_etmek              67
  9_ozur_dilemek            68

VAL — 282 vide

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Nöron-Ağları-Proje/Train/train_labels.csv')
print(df.head(10))
print(f"\nToplam örnek: {len(df)}")
print(f"Toplam sınıf: {df.iloc[:, 1].nunique()}")

    signer0_sample1   41
0   signer0_sample2  104
1   signer0_sample3  205
2   signer0_sample4   26
3   signer0_sample5  191
4   signer0_sample6  140
5   signer0_sample7  139
6   signer0_sample8  225
7   signer0_sample9  102
8  signer0_sample10   19
9  signer0_sample11  160

Toplam örnek: 28141
Toplam sınıf: 226


In [ ]:
# Bizim 30 sınıfımızın isimleri
our_classes = ['0_ben', '1_sen', '2_selam', '3_hoscakal', '4_tamam',
               '5_evet', '6_hayir', '7_tesekkur', '8_rica_etmek', '9_ozur_dilemek',
               '10_var', '11_yok', '12_iyi', '13_kotu', '14_nasil',
               '15_neden', '16_nerede', '17_yardim', '18_doktor', '19_hasta',
               '20_hastane', '21_ilac', '22_gecmis_olsun', '23_ataturk', '24_ev',
               '25_zaman', '26_icmek', '27_yemek', '28_yapmak', '29_bakmak']

# Sayısal etiketleri çıkar (klasör adındaki sayı)
our_label_ids = [int(c.split('_')[0]) for c in our_classes]
print("Bizim sınıf ID'leri:", sorted(our_label_ids))

# Bu ID'lere sahip kaç örnek var?
df.columns = ['filename', 'label']
for label_id, class_name in zip(our_label_ids, our_classes):
    count = (df['label'] == label_id).sum()
    print(f"{class_name:25s} → label {label_id:3d} → {count} örnek")

Bizim sınıf ID'leri: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
0_ben                     → label   0 → 126 örnek
1_sen                     → label   1 → 125 örnek
2_selam                   → label   2 → 123 örnek
3_hoscakal                → label   3 → 123 örnek
4_tamam                   → label   4 → 125 örnek
5_evet                    → label   5 → 125 örnek
6_hayir                   → label   6 → 125 örnek
7_tesekkur                → label   7 → 122 örnek
8_rica_etmek              → label   8 → 125 örnek
9_ozur_dilemek            → label   9 → 122 örnek
10_var                    → label  10 → 121 örnek
11_yok                    → label  11 → 126 örnek
12_iyi                    → label  12 → 127 örnek
13_kotu                   → label  13 → 124 örnek
14_nasil                  → label  14 → 119 örnek
15_neden                  → label  15 → 121 örnek
16_nerede                 → label  16 → 108 örnek
17_yardim         

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/content')
print(f"Toplam : {total/1e9:.1f} GB")
print(f"Kullanılan: {used/1e9:.1f} GB")
print(f"Boş   : {free/1e9:.1f} GB")

Toplam : 120.9 GB
Kullanılan: 46.0 GB
Boş   : 74.9 GB


In [ ]:
import subprocess

# validation_labels.zip'i aç
subprocess.run([
    'unzip', '-P', 'zYX5W7fZ', '-q',
    '/content/drive/MyDrive/Nöron-Ağları-Proje/validation/validation_labels.zip',
    '-d', '/content/labels/'
], capture_output=True)

# Dosyayı bul ve oku
import os
for f in os.listdir('/content/labels/'):
    print(f)

ground_truth.csv


In [ ]:
df_val = pd.read_csv('/content/labels/ground_truth.csv',
                     header=None, names=['filename', 'label'])
print(f"Val toplam örnek: {len(df_val)}")
print(f"Val toplam sınıf: {df_val['label'].nunique()}")
print()

print("--- VAL (bizim 30 sınıf) ---")
for label_id, class_name in zip(our_label_ids, our_classes):
    count = (df_val['label'] == label_id).sum()
    print(f"{class_name:25s} → {count} örnek")

Val toplam örnek: 4418
Val toplam sınıf: 226

--- VAL (bizim 30 sınıf) ---
0_ben                     → 20 örnek
1_sen                     → 19 örnek
2_selam                   → 20 örnek
3_hoscakal                → 19 örnek
4_tamam                   → 20 örnek
5_evet                    → 20 örnek
6_hayir                   → 13 örnek
7_tesekkur                → 20 örnek
8_rica_etmek              → 20 örnek
9_ozur_dilemek            → 19 örnek
10_var                    → 19 örnek
11_yok                    → 20 örnek
12_iyi                    → 19 örnek
13_kotu                   → 20 örnek
14_nasil                  → 20 örnek
15_neden                  → 20 örnek
16_nerede                 → 20 örnek
17_yardim                 → 18 örnek
18_doktor                 → 17 örnek
19_hasta                  → 19 örnek
20_hastane                → 19 örnek
21_ilac                   → 19 örnek
22_gecmis_olsun           → 20 örnek
23_ataturk                → 20 örnek
24_ev                     → 19 örnek


In [ ]:
# Test labels
subprocess.run([
    'unzip', '-P', 'ds6Kvdus3o', '-q',
    '/content/drive/MyDrive/Nöron-Ağları-Proje/Test/test_labels.zip',
    '-d', '/content/labels/'
], capture_output=True)

import os
for f in os.listdir('/content/labels/'):
    print(f)

ground_truth.csv


In [ ]:
# Önce val'ı farklı isimle kaydet, sonra test'i aç
import shutil
shutil.copy('/content/labels/ground_truth.csv', '/content/labels/val_ground_truth.csv')

# Test'i farklı dizine aç
os.makedirs('/content/labels/test/', exist_ok=True)
subprocess.run([
    'unzip', '-P', 'ds6Kvdus3o', '-q',
    '/content/drive/MyDrive/Nöron-Ağları-Proje/Test/test_labels.zip',
    '-d', '/content/labels/test/'
], capture_output=True)

df_test = pd.read_csv('/content/labels/test/ground_truth.csv',
                      header=None, names=['filename', 'label'])
print(f"Test toplam örnek: {len(df_test)}")
print(f"Test toplam sınıf: {df_test['label'].nunique()}")
print()

print("--- TEST (bizim 30 sınıf) ---")
for label_id, class_name in zip(our_label_ids, our_classes):
    count = (df_test['label'] == label_id).sum()
    print(f"{class_name:25s} → {count} örnek")

Test toplam örnek: 3742
Test toplam sınıf: 226

--- TEST (bizim 30 sınıf) ---
0_ben                     → 16 örnek
1_sen                     → 16 örnek
2_selam                   → 17 örnek
3_hoscakal                → 17 örnek
4_tamam                   → 17 örnek
5_evet                    → 17 örnek
6_hayir                   → 17 örnek
7_tesekkur                → 17 örnek
8_rica_etmek              → 17 örnek
9_ozur_dilemek            → 17 örnek
10_var                    → 17 örnek
11_yok                    → 16 örnek
12_iyi                    → 15 örnek
13_kotu                   → 16 örnek
14_nasil                  → 17 örnek
15_neden                  → 15 örnek
16_nerede                 → 14 örnek
17_yardim                 → 16 örnek
18_doktor                 → 17 örnek
19_hasta                  → 17 örnek
20_hastane                → 16 örnek
21_ilac                   → 17 örnek
22_gecmis_olsun           → 17 örnek
23_ataturk                → 16 örnek
24_ev                     → 16 örn

In [ ]:
result = subprocess.run(
    ['7z', 'l', '-pMdG3z6Eh1t',
     '/content/drive/MyDrive/Nöron-Ağları-Proje/Train/train_set_vfbha39.zip.001'],
    capture_output=True, text=True
)

# İlk 50 satırı göster
lines = result.stdout.split('\n')
for line in lines[:80]:
    print(line)


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,12 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (50657),ASM,AES-NI)

Scanning the drive for archives:
1 file, 1048576000 bytes (1000 MiB)

Listing archive: /content/drive/MyDrive/Nöron-Ağları-Proje/Train/train_set_vfbha39.zip.001

--
Path = /content/drive/MyDrive/Nöron-Ağları-Proje/Train/train_set_vfbha39.zip.001
Type = Split
Physical Size = 1048576000
Volumes = 18
Total Physical Size = 18514780403
----
Path = train_set_vfbha39.zip
Size = 18514780403
--
Path = train_set_vfbha39.zip
Type = zip
Physical Size = 18514780403
64-bit = +

   Date      Time    Attr         Size   Compressed  Name
------------------- ----- ------------ ------------  ------------------------
2020-12-18 20:46:16 D....            0            0  train
2020-12-18 20:44:48 ....A       355277       353725  train/signer0_sample1000_color.mp4
2020-12-18 20:44:48 ....A       154996       1419

In [ ]:
result = subprocess.run(
    ['7z', 'l', '-pMdG3z6Eh1t',
     '/content/drive/MyDrive/Nöron-Ağları-Proje/Train/train_set_vfbha39.zip.001'],
    capture_output=True, text=True
)

# Unique uzantıları bul
extensions = set()
for line in result.stdout.split('\n'):
    if '....A' in line:
        filename = line.strip().split()[-1]
        ext = filename.split('.')[-1]
        extensions.add(ext)

print("Zip'teki dosya uzantıları:", extensions)

Zip'teki dosya uzantıları: {'mp4'}


## **Veri Çıkarımı**

# TİD Özellik Çıkarımı

**Çıkarılan özellikler (170 toplam):**
- Pose: 11 nokta × 4 = 44 (burun, kulaklar, omuzlar, dirsekler, bilekler, kalçalar)
- Sol el: 21 × 3 = 63
- Sağ el: 21 × 3 = 63

**Çıktı:** Her video → (T, 170) shape .npy dosyası  
**Kayıt:** Drive/ErenDataset/train|val|test/sınıf_adı/

In [ ]:
%pip install mediapipe==0.10.14 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 28.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.


In [ ]:
import sys
sys.modules['tensorflow'] = None

import mediapipe as mp
mp_holistic = mp.solutions.holistic
print("Calisiyor")

Calisiyor


In [ ]:
!sudo apt-get install p7zip-full -y -q

Reading package lists...
Building dependency tree...
Reading state information...
p7zip-full is already the newest version (16.02+dfsg-8).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import mediapipe as mp
import cv2
import numpy as np
import os
import subprocess
import pandas as pd
from tqdm.notebook import tqdm

DRIVE_ROOT  = '/content/drive/MyDrive/Nöron-Ağları-Proje'
OUTPUT_ROOT = f'{DRIVE_ROOT}/ErenDataset'

TRAIN_ZIP  = f'{DRIVE_ROOT}/Train/train_set_vfbha39.zip.001'
VAL_ZIP    = f'{DRIVE_ROOT}/validation/val_set_bjhfy68.zip.001'
TEST_ZIP   = f'{DRIVE_ROOT}/Test/test_set_xsaft57.zip.001'

TRAIN_PASS = 'MdG3z6Eh1t'
VAL_PASS   = 'bhRY5B9zS2'
TEST_PASS  = 'ds6Kvdus3o'

POSE_LANDMARKS = [0, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24]

our_classes = [
    '0_ben', '1_sen', '2_selam', '3_hoscakal', '4_tamam',
    '5_evet', '6_hayir', '7_tesekkur', '8_rica_etmek', '9_ozur_dilemek',
    '10_var', '11_yok', '12_iyi', '13_kotu', '14_nasil',
    '15_neden', '16_nerede', '17_yardim', '18_doktor', '19_hasta',
    '20_hastane', '21_ilac', '22_gecmis_olsun', '23_ataturk', '24_ev',
    '25_zaman', '26_icmek', '27_yemek', '28_yapmak', '29_bakmak'
]
our_label_ids = set(range(30))

print('Sabitler tanımlandı')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sabitler tanımlandı


In [ ]:
import os

# Test klasörü
test_dir = f'{DRIVE_ROOT}/Test'
print('TEST klasörü:')
for f in sorted(os.listdir(test_dir)):
    size = os.path.getsize(os.path.join(test_dir, f)) / (1024**2)
    print(f'  {f:40s} {size:.1f} MB')

print('\nVALIDATION klasörü:')
val_dir = f'{DRIVE_ROOT}/validation'
for f in sorted(os.listdir(val_dir)):
    size = os.path.getsize(os.path.join(val_dir, f)) / (1024**2)
    print(f'  {f:40s} {size:.1f} MB')

print('\nTRAIN klasörü:')
train_dir = f'{DRIVE_ROOT}/Train'
for f in sorted(os.listdir(train_dir)):
    size = os.path.getsize(os.path.join(train_dir, f)) / (1024**2)
    print(f'  {f:40s} {size:.1f} MB')

TEST klasörü:
  test_labels.zip                          0.0 MB
  test_set_xsaft57.zip.001                 1000.0 MB
  test_set_xsaft57.zip.002                 1000.0 MB
  test_set_xsaft57.zip.003                 569.0 MB

VALIDATION klasörü:
  val_set_bjhfy68.zip.001                  1000.0 MB
  val_set_bjhfy68.zip.002                  1000.0 MB
  val_set_bjhfy68.zip.003                  981.0 MB
  validation_labels.zip                    0.0 MB

TRAIN klasörü:
  train_labels.csv                         0.6 MB
  train_set_vfbha39.zip.001                1000.0 MB
  train_set_vfbha39.zip.002                1000.0 MB
  train_set_vfbha39.zip.003                1000.0 MB
  train_set_vfbha39.zip.004                1000.0 MB
  train_set_vfbha39.zip.005                1000.0 MB
  train_set_vfbha39.zip.006                1000.0 MB
  train_set_vfbha39.zip.007                1000.0 MB
  train_set_vfbha39.zip.008                1000.0 MB
  train_set_vfbha39.zip.009                1000.0 MB
  trai

In [ ]:
test_cases = [
    ('TRAIN video  ', f'{DRIVE_ROOT}/Train/train_set_vfbha39.zip.001',      'MdG3z6Eh1t'),
    ('VAL video    ', f'{DRIVE_ROOT}/validation/val_set_bjhfy68.zip.001',   'zYX5W7fZ'),
    ('VAL labels   ', f'{DRIVE_ROOT}/validation/validation_labels.zip',     'zYX5W7fZ'),
    ('TEST video   ', f'{DRIVE_ROOT}/Test/test_set_xsaft57.zip.001',        'ds6Kvdus3o'),
    ('TEST labels  ', f'{DRIVE_ROOT}/Test/test_labels.zip',                 'ds6Kvdus3o'),
]

for name, path, pwd in test_cases:
    result = subprocess.run(
        ['7z', 't', f'-p{pwd}', path],
        capture_output=True, text=True
    )
    has_error = 'Sub items Errors' in result.stdout
    print(f'{name} | {pwd} | {"YANLIS" if has_error else "DOGRU"}')
    if has_error:
        # Hata sayısını göster
        for line in result.stdout.split('\n'):
            if 'Errors' in line:
                print(f'         {line.strip()}')

TRAIN video   | MdG3z6Eh1t | YANLIS
         Sub items Errors: 2
         Archives with Errors: 1
         Sub items Errors: 2
VAL video     | zYX5W7fZ | YANLIS
         Sub items Errors: 8836
         Archives with Errors: 1
         Sub items Errors: 8836
VAL labels    | zYX5W7fZ | DOGRU
TEST video    | ds6Kvdus3o | YANLIS
         Sub items Errors: 1
         Archives with Errors: 1
         Open Errors: 1
         Sub items Errors: 1
TEST labels   | ds6Kvdus3o | DOGRU


In [ ]:
result = subprocess.run(
    ['7z', 't', '-pbhRY5B9zS2',
     f'{DRIVE_ROOT}/validation/val_set_bjhfy68.zip.001'],
    capture_output=True, text=True
)
has_error = 'Sub items Errors' in result.stdout
print(f'Val | bhRY5B9zS2 | {"YANLIS" if has_error else "DOGRU"}')
for line in result.stdout.split('\n'):
    if 'Errors' in line or 'Ok' in line:
        print(f'  {line.strip()}')

Val | bhRY5B9zS2 | DOGRU
  Everything is Ok


In [ ]:
import subprocess
import os

os.makedirs('/content/labels/test', exist_ok=True)

# Val labels
subprocess.run([
    'unzip', '-P', 'zYX5W7fZ', '-q',
    f'{DRIVE_ROOT}/validation/validation_labels.zip',
    '-d', '/content/labels/'
], capture_output=True)

import shutil
shutil.copy('/content/labels/ground_truth.csv', '/content/labels/val_ground_truth.csv')

# Test labels
subprocess.run([
    'unzip', '-P', 'ds6Kvdus3o', '-q',
    f'{DRIVE_ROOT}/Test/test_labels.zip',
    '-d', '/content/labels/test/'
], capture_output=True)

print("Label dosyalari hazir")

Label dosyalari hazir


In [ ]:
# Train labels
df_train = pd.read_csv(f'{DRIVE_ROOT}/Train/train_labels.csv',
                       header=None, names=['filename', 'label'])

# Val labels
df_val = pd.read_csv('/content/labels/val_ground_truth.csv',
                     header=None, names=['filename', 'label'])

# Test labels
df_test = pd.read_csv('/content/labels/test/ground_truth.csv',
                      header=None, names=['filename', 'label'])

print(f'Train: {(df_train.label.isin(our_label_ids)).sum()} video')
print(f'Val  : {(df_val.label.isin(our_label_ids)).sum()} video')
print(f'Test : {(df_test.label.isin(our_label_ids)).sum()} video')

Train: 3696 video
Val  : 578 video
Test : 494 video


In [ ]:
mp_holistic = mp.solutions.holistic

def extract_keypoints(frame_bgr, holistic):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    results = holistic.process(rgb)

    # Pose
    if results.pose_landmarks:
        pose = np.array([
            [results.pose_landmarks.landmark[i].x,
             results.pose_landmarks.landmark[i].y,
             results.pose_landmarks.landmark[i].z,
             results.pose_landmarks.landmark[i].visibility]
            for i in POSE_LANDMARKS
        ]).flatten()
    else:
        pose = np.zeros(len(POSE_LANDMARKS) * 4)

    # Sol el
    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z]
                       for lm in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21 * 3)

    # Sağ el
    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z]
                       for lm in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21 * 3)

    return np.concatenate([pose, lh, rh])  # (170,)


def process_split(zip_path, zip_password, label_df, split_name):
    print(f'\n{split_name.upper()} işleniyor...')

    extract_dir = f'/content/extracted_{split_name}'
    os.makedirs(extract_dir, exist_ok=True)
    output_dir  = os.path.join(OUTPUT_ROOT, split_name)

    for cls in our_classes:
        os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

    label_map    = dict(zip(label_df['filename'], label_df['label']))
    target_files = {f: l for f, l in label_map.items() if l in our_label_ids}

    success, fail = 0, 0

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        for fname, label in tqdm(sorted(target_files.items()),
                                 desc=split_name, unit='video'):
            cls_name   = our_classes[label]
            video_name = f'{fname}_color.mp4'
            zip_path_internal = f'{split_name}/{video_name}'
            local_video = os.path.join(extract_dir, video_name)
            output_npy  = os.path.join(output_dir, cls_name, f'{fname}_color.npy')

            if os.path.exists(output_npy):
                success += 1
                continue

            subprocess.run(
                ['7z', 'e', f'-p{zip_password}', zip_path,
                 zip_path_internal, f'-o{extract_dir}', '-y'],
                capture_output=True
            )

            if not os.path.exists(local_video):
                fail += 1
                continue

            cap    = cv2.VideoCapture(local_video)
            frames = []
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                frames.append(extract_keypoints(frame, holistic))
            cap.release()
            os.remove(local_video)

            if len(frames) == 0:
                fail += 1
                continue

            np.save(output_npy, np.array(frames, dtype=np.float32))
            success += 1

    print(f'{split_name.upper()} bitti — {success} başarılı, {fail} başarısız')

print('Fonksiyonlar hazır')

Fonksiyonlar hazır


In [ ]:
process_split(TRAIN_ZIP, TRAIN_PASS, df_train, 'train')


TRAIN işleniyor...


train:   0%|          | 0/3696 [00:00<?, ?video/s]

TRAIN bitti — 3696 başarılı, 0 başarısız


In [ ]:
process_split(VAL_ZIP, VAL_PASS, df_val, 'val')


VAL işleniyor...


val:   0%|          | 0/578 [00:00<?, ?video/s]

VAL bitti — 578 başarılı, 0 başarısız


In [ ]:
process_split(TEST_ZIP, TEST_PASS, df_test, 'test')


TEST işleniyor...


test:   0%|          | 0/494 [00:00<?, ?video/s]

TEST bitti — 494 başarılı, 0 başarısız


In [ ]:
import glob

for split in ['train', 'val', 'test']:
    files = glob.glob(f'{OUTPUT_ROOT}/{split}/**/*.npy', recursive=True)
    print(f'{split}: {len(files)} dosya')

train: 3696 dosya
val: 578 dosya
test: 494 dosya
